In [ ]:
import torch
import numpy as np
from torchvision.transforms import functional as funt
import matplotlib.pyplot as plt
import skimage
from skimage import io
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms.functional as F
from copy import deepcopy
from torch.utils.data import DataLoader
import torch.optim as optim
from sklearn.model_selection import train_test_split

In [ ]:
images = io.imread("")
images.shape

In [ ]:
# @title Wavelet Filtering
# kernels for atrous filter
def aTrousKernel(J):
  nzeros = 2**(J-1) - 1
  fsize = 4 * nzeros + 5
  filter = torch.zeros(fsize)
  filter[0] = 1.0 / 16.0
  filter[1 + nzeros] = 1.0 / 4.0
  filter[2 * (1 + nzeros)] = 3.0 / 8.0
  filter[3 * (1 + nzeros)] = 1.0 / 4.0
  filter[fsize - 1] = 1.0 / 16.0
  return torch.outer(filter, filter).reshape(1, 1, fsize, fsize)

J = 5 # number of scales
scales_list = []

# generate the pipeline
for jj in range(J):
  atrous_kernel = aTrousKernel(jj + 1)
  scale = nn.Conv2d(1, 1, atrous_kernel.shape[2], padding = atrous_kernel.shape[2] // 2, bias = False, padding_mode='reflect')
  scale.weight = torch.nn.Parameter(atrous_kernel)
  # remove grads
  for _ in scale.parameters():
    _.requires_grad_(False)
    #print(_)
  scales_list.append(scale)

# process images
imageW = []
im_in = torch.from_numpy(images).float().unsqueeze(1)
for jj in range(J):
  sc = scales_list[jj](im_in)
  imageW.append(im_in - sc)
  im_in = sc

imageW.append(im_in) # append last 'background' image in the end of the list

imageW[0][imageW[0] < 3 * imageW[0].std(dim=(1,2,3), keepdim=True)] = 0
imageW[1][imageW[1] < 3 * imageW[1].std(dim=(1,2,3), keepdim=True)] = 0
imageW[2][imageW[2] < 3 * imageW[2].std(dim=(1,2,3), keepdim=True)] = 0
imageW[3][imageW[3] < 3 * imageW[3].std(dim=(1,2,3), keepdim=True)] = 0
imageW[4][imageW[4] < 3 * imageW[4].std(dim=(1,2,3), keepdim=True)] = 0

# create fake image with 4 scales
wavelet_images = torch.cat((imageW[1], imageW[2], imageW[3], imageW[4]), axis = 1)
#wavelet_images = torch.cat((torch.from_numpy(wavelet_images), torch.from_numpy(wavelet_images), torch.from_numpy(wavelet_images)), axis = 1)

In [ ]:
wavelet_images[id,0].std().item()*3

In [ ]:
id = 0
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize = (16, 5))
ax1.imshow(wavelet_images[id,0], cmap = 'gray', vmax = np.percentile(images[id,0], 99) )
ax2.imshow(wavelet_images[id,1], cmap = 'gray', vmax = np.percentile(images[id,1], 99))
ax3.imshow(wavelet_images[id,2], cmap = 'gray', vmax = np.percentile(images[id,2], 99))
# ax1.imshow(wavelet_images[id,0], cmap = 'gray', vmin=-wavelet_images[id,0].std().item()*3*, vmax=wavelet_images[id,0].std().item()*3)
# ax2.imshow(wavelet_images[id,1], cmap = 'gray',  vmin=-wavelet_images[id,1].std().item()*3, vmax=wavelet_images[id,1].std().item()*3)
# ax3.imshow(wavelet_images[id,2], cmap = 'gray',  vmin=-wavelet_images[id,2].std().item()*3, vmax=wavelet_images[id,2].std().item()*3)

# ax4.imshow(wavelet_images[id,3], cmap = 'gray',  vmin=-wavelet_images[id,3].std().item()*3, vmax=wavelet_images[id,3].std().item()*3)
#labels[id]

In [ ]:
plt.imshow(imageW[4][id,0])

In [ ]:
plt.imshow(imageW[1][id,0] + imageW[2][id,0], vmin=0, vmax = 3000)

Save the wavelet-filtered images

In [ ]:
import os
import skimage.io as io

# Ensure the output directory exists
output_dir = ""
os.makedirs(output_dir, exist_ok=False)

# Iterate through all image IDs
for img_id in range(wavelet_images.shape[0]): # wavelet_images.shape[0] is the total number of images
    # Get all 4 wavelet scales (channels) for the current image ID
    # The shape will be (num_channels, height, width), e.g., (4, 256, 256)
    image_channels = wavelet_images[img_id].cpu().numpy()

    # Transpose the array to (height, width, num_channels) for multi-channel saving
    # This matches the expected format for skimage.io.imsave for multi-channel images
    image_to_save = image_channels.transpose((1, 2, 0))

    # Define filename for the multi-channel TIFF
    filename = os.path.join(output_dir, f"wavelet_multi_channel_id_{img_id}.tif")

    # Save the multi-channel image as TIFF to preserve float data
    io.imsave(filename, image_to_save)
    # print(f"Saved {filename}") # Optional: uncomment to print every saved file

print(f"All multi-channel wavelet images for all IDs saved to {output_dir}")